<a href="https://colab.research.google.com/github/mille-s/GEM24_EvalLLM/blob/main/GEM24_EvalLLM_OpenAI_SM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Please go to this github for more information on the result analysis: https://github.com/mille-s/GEM24_EvalLLM

In [66]:
#@title Install OpenAI
from IPython.display import clear_output
# ! pip install aixplain
# ! pip install openai==0.28
# ! pip install groq
# ! pip install anthropic
# !pip install --upgrade openai

clear_output()

In [67]:
import json
import codecs
from bs4 import BeautifulSoup
import os
import time
import csv
import pandas as pd
import pickle
import glob
import re
from typing import Dict, Any

In [68]:
prompt = '''
In this task, you will evaluate the quality of the Text in relation to the given Triple Set. How well does the Text represent the Triple Set?  You will be given four specific Dimensions to evaluate against:

Dimensions:"""
No-Omissions: ALL the information in the Triple Set is present in the Text.
No-Additions: ONLY information from the Triple Set is present in the Text.
Grammaticality: The Text is free of grammatical and spelling errors.
Fluency: The Text flows well and is easy to read; its parts are connected in a natural way."""

Important note on No-Omissions and No-Additions: some Triple Set/Text pairs contain non-factual information and even fictional names for people, places, dates, etc. Whether there are omissions and/or additions in a Text is NOT related to factual truth, but instead is strictly related to the contents of the input Triple Set.
Important note on Grammaticality and Fluency: for Grammaticality and Fluency you do not need to consider the input Triple Set; only the intrinsic quality of the Text needs to be assessed.

You need to provide the scores ranging from 1 (indicating the lowest score) to 7 (indicating the highest score) for each of the dimensions and a short justification for each score in the following JSON format:  {{"No-Omissions": {{"Justification": "", "Score": ""}}, "No-Additions": {{"Justification": "", "Score": ""}}, "Grammaticality": {{"Justification": "", "Score": ""}}, "Fluency": {{"Justification": "", "Score": ""}} }}.

Make sure to read thoroughly the Triple Set and the {language} Text below, and assess the four Dimensions using the instructions and template above.

Triple Set: {Triples} \nText: {Nice_Text} \n\n
'''

In [70]:
class ModelEvaluator:
    # With aiXplain: llama32 = "6704c91bfdf7d14548c9fedb"; deepseekr1 = "67976f47e341d313f66bb835"
    MODEL_REGISTRY = {
        "o3": {"type": "openai", "folder": "GPT_results"},
        "deepseek-r1-distill-llama-70b": {"type": "groq", "folder": "GroqDeepseek_results"},
        "claude-3-7-sonnet-latest": {"type": "anthropic", "folder": "Claude_results"},
        "Gemini_1dot5_Flash": {"type": "aixplain", "folder": "Gemini_results"},
        "6704c91bfdf7d14548c9fedb": {"type": "aixplain", "folder": "aiXplainLlama_results"},
        "67976f47e341d313f66bb835": {"type": "aixplain", "folder": "aiXplainDeepseek_results"},
        # Add more as needed
    }

    def __init__(self):
        pass

    def _call_openai(self, prompt, model):
        import openai
        response = openai.ChatCompletion.create(
            model=model,
            messages=[{"role": "system", "content": prompt}],
            temperature=1
        )
        return response['choices'][0]['message']['content']

    def _call_groq(self, prompt, model):
        from groq import Groq
        client = Groq()
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            model=model,
            temperature=1,
        )
        return response.choices[0].message.content

    def _call_aixplain(self, prompt, model):
        from aixplain.factories import AgentFactory
        from aixplain.modules.agent.tool.model_tool import ModelTool
        agent = AgentFactory.create(
            name="Assessment of text quality",
            description="Assessment of text quality",
            instructions="",
            tools=[ModelTool(model=model)],
        )
        return agent.run(prompt)

    def _call_anthropic(self, prompt, model):
        import anthropic
        client = anthropic.Anthropic()
        message = client.messages.create(
            model=model,
            max_tokens=1000,
            temperature=1,
            system="You are a helpful assistant.",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt}
                    ]
                }
            ]
        )
        if isinstance(message.content, list):
            text = "".join([tb.text for tb in message.content if hasattr(tb, "text")])
            return text
        return str(message.content)

    def runEval(self, triples_text_pairs, model, prompt_template, n_examples=None, language="English"):
        model_info = self.MODEL_REGISTRY.get(model)
        if not model_info:
            raise ValueError(f"Unknown model: {model}")

        out_folder = os.path.join(language, model_info['folder'])
        os.makedirs(out_folder, exist_ok=True)
        backend = model_info["type"]

        # Find already processed IDs
        existing_files = os.listdir(out_folder)
        processed_ids = set()
        prefix = f'{model}_results_'  # Handles all models generically
        for fname in existing_files:
            if fname.startswith(prefix):
                processed_ids.add(fname[len(prefix):].replace('.pkl', ''))

        if n_examples is None:
            n_examples = len(triples_text_pairs)

        for i, dp in enumerate(triples_text_pairs[:n_examples]):
            ex_id = str(dp['id'])  # Always treat as str for consistency

            if ex_id in processed_ids:
                # print(f"Skipping text #{i} (ID={ex_id}) -- already processed.")
                continue

            Triples = dp['triples']
            Nice_Text = dp['text']
            # prompt = prompt_template.format(language=language, Triples=Triples, Nice_Text=Nice_Text)
            prompt = '''
In this task, you will evaluate the quality of the Text in relation to the given Triple Set. How well does the Text represent the Triple Set?  You will be given four specific Dimensions to evaluate against:

Dimensions:"""
No-Omissions: ALL the information in the Triple Set is present in the Text.
No-Additions: ONLY information from the Triple Set is present in the Text.
Grammaticality: The Text is free of grammatical and spelling errors.
Fluency: The Text flows well and is easy to read; its parts are connected in a natural way."""

Important note on No-Omissions and No-Additions: some Triple Set/Text pairs contain non-factual information and even fictional names for people, places, dates, etc. Whether there are omissions and/or additions in a Text is NOT related to factual truth, but instead is strictly related to the contents of the input Triple Set.
Important note on Grammaticality and Fluency: for Grammaticality and Fluency you do not need to consider the input Triple Set; only the intrinsic quality of the Text needs to be assessed.

You need to provide the scores ranging from 1 (indicating the lowest score) to 7 (indicating the highest score) for each of the dimensions and a short justification for each score in the following JSON format:  {"No-Omissions": {"Justification": "", "Score": ""}, "No-Additions": {"Justification": "", "Score": ""}, "Grammaticality": {"Justification": "", "Score": ""}, "Fluency": {"Justification": "", "Score": ""} }.

Make sure to read thoroughly the Triple Set and the '''+str(language)+''' Text below, and assess the four Dimensions using the instructions and template above.

Triple Set: ''' + str(Triples) + "\n" + '''Text: '''+ str(Nice_Text) + "\n\n" + '''
'''
            print(f"Evaluating #{i+1}/{n_examples} for model {model}...")

            # Call the right backend
            try:
                if backend == "openai":
                    response = self._call_openai(prompt, model)
                elif backend == "groq":
                    response = self._call_groq(prompt, model)
                elif backend == "aixplain":
                    response = self._call_aixplain(prompt, model)
                elif backend == "anthropic":
                    response = self._call_anthropic(prompt, model)
                else:
                    raise ValueError(f"Unsupported backend: {backend}")
            except Exception as e:
                print(f"Error on example {i} ({ex_id}): {e}")
                response = None
            
            # if "</think>" in response:
            #     response = response.split("</think>")[-1].strip()

            # Store the result
            dp[f'scores_{model}'] = response
            with open(os.path.join(out_folder, f'{model}_results_{ex_id}'), 'ab') as f:
                pickle.dump(dp, f)

            time.sleep(10)  # Sleep to avoid rate limits

        print(f"Results for {model} saved in {out_folder}/")
        # return triples_text_pairs


In [71]:
#@title Download and Load human-eval-packaged json, and format contents (triples, text, id)
def format_json(json_path):
  en_regular_json = json.load(codecs.open(json_path, 'r', 'utf-8'))
  triples_text_pairs = []
  x = 0
  while x < len(en_regular_json):
    # Parse html found in the "input" key
    html = en_regular_json[x]['input']
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table')
    rows = []
    for row in table.find_all('tr'):
      columns = row.find_all(['td', 'th'])  # Get both <td> and <th>
      row_data = ' '.join([col.text.strip() for col in columns])
      rows.append(row_data)
    triples_formatted = '; '.join(rows[1:]) # exclude header
    triples_text_pairs.append({'id':en_regular_json[x]['id'], 'triples': '"""'+triples_formatted+'"""', 'text': en_regular_json[x]['output']})
    x += 1
  return triples_text_pairs

In [72]:
#@title Load Custom json file

custom_filepath = 'en_longInputD2T_LLMtexts.json'
language = None
if custom_filepath.endswith('en_longInputD2T_LLMtexts.json'):
  language = 'English'
elif custom_filepath.endswith('ga_longInputD2T_LLMtexts.json'):
  language = 'Irish'

print(f'Language: {language}')
triples_text_pairs = format_json(custom_filepath)
print(f'{len(triples_text_pairs)} datapoints found!')
# print(triples_text_pairs[0])

Language: English
210 datapoints found!


In [75]:
#@param["03", "claude-3-7-sonnet-latest", "deepseek-r1-distill-llama-70b", "Llama 3.2 3B Instruct"]
# With aiXplain: llama32 = "6704c91bfdf7d14548c9fedb"; deepseekr1 = "67976f47e341d313f66bb835"
# with groq: model= "deepseek-r1-distill-llama-70b"
# with openai: model = "o3"


# model="deepseek-r1-distill-llama-70b"
# model="6704c91bfdf7d14548c9fedb"
model="6704c91bfdf7d14548c9fedb"
# model="claude-3-7-sonnet-latest"
# model="o3" 
n_examples = None #len(triples_text_pairs)

In [ ]:
evaluator = ModelEvaluator()
evaluator.runEval(triples_text_pairs, model=model, prompt_template=prompt, n_examples=n_examples, language=language)

## Results analysis

In [ ]:
# #@title Load unzipped files
# import pickle
# import glob
# import json
# import re
# import codecs
# import os
# import ast

# def auto_close_json(s):
#     # Counts how many more } are needed
#     open_braces = s.count('{')
#     close_braces = s.count('}')
#     needed = open_braces - close_braces
#     if needed > 0:
#         s += '}' * needed
#     return s

# update_params_unzip = True #@param {type:"boolean"}
# if update_params_unzip:
#   zip_language = 'EN' #@param['EN', 'ES', 'SW']
#   zip_model = 'o3' #@param['GPT-4o-mini', 'GPT-o3-mini', 'Gemini-1dot5-flash']
#   zip_data = 'regular' #@param['regular', 'iaa']

# model_scores = [1, 2, 3, 4, 5, 6, 7]
# # load_gemini_folder = True #@param {type:"boolean"}
# # load_gpt_folder = False #@param {type:"boolean"}
# path_dir_unzipped = ''
# model_prefix = ''
# if zip_model.startswith('Gemini'):
#   path_dir_unzipped = os.path.join('/content', zip_language+'_'+zip_data, zip_model, 'content', 'Gemini_results')
#   model_prefix = 'Gemini'
# # elif zip_model.startswith('GPT'):
# else:
#   path_dir_unzipped = os.path.join('GPT_results')
#   model_prefix = 'GPT'

# # def separateJustification(LLMoutString, criterion):
# #   """
# #   The Justifications returned by the models often break the json format, so I extract them
# #   """
# #   search_expression = '("'+criterion+'":[^\{]+\{[^\}]*"Justification":)([^\}]+)("Score":[^\}]+\})'
# #   if re.search(search_expression, LLMoutString):
# #     justificationRemoved = re.sub(search_expression, '\g<1> "", \g<3>',  LLMoutString)
# #     justification = re.sub('^.*'+search_expression+'.*$', '\g<2>',  LLMoutString)
# #   else:
# #     justificationRemoved = LLMoutString
# #     justification = ''
# #   return justificationRemoved, justification

# def separateJustification(LLMoutString, criterion):
#     """
#     The Justifications returned by the models often break the json format, so I extract them
#     """
#     # Use raw strings for regex, and build using f-strings for clarity.
#     search_expression = (
#         rf'("{re.escape(criterion)}":[^\{{]+\{{[^\}}]*"Justification":)([^\}}]+)("Score":[^\}}]+\}})'
#     )
#     if re.search(search_expression, LLMoutString):
#         justificationRemoved = re.sub(search_expression, r'\g<1> "", \g<3>', LLMoutString)
#         justification = re.sub(rf'^.*{search_expression}.*$', r'\g<2>', LLMoutString)
#     else:
#         justificationRemoved = LLMoutString
#         justification = ''
#     return justificationRemoved, justification


# def loadDataPoint(dbfile_x, model):
#   eval_missing = None
#   wrong_score = None
#   dico_key = 'scores_'+str(model)
#   formatted_scores = {}
#   # load data with pickle
#   dp = pickle.load(dbfile_x)
#   if dico_key in dp:
#     print(dp['id'])
#     # print(dp['triples'])
#     # print(dp['text'])
#     justifications = []
#     # pickle.load uses single quotes, whereas json.load expects double quotes
#     # Gemini adds a node "query" in the json, unlike OpenAI's models
#     LLMout_string = str(dp[dico_key]).replace("'query'", '"query"')
#     LLMout_string = LLMout_string.replace("```json", "")
#     LLMout_string = LLMout_string.replace("```", "")
#     LLMout_string = LLMout_string.replace("'No-Omissions'", '"No-Omissions"')
#     # There's a typo in one of the Gemini outputs
#     LLMout_string = LLMout_string.replace("'No-Omissons'", '"No-Omissions"')
#     LLMout_string = LLMout_string.replace('"No-Omissons"', '"No-Omissions"')
#     LLMout_string = LLMout_string.replace("'No-Additions'", '"No-Additions"')
#     LLMout_string = LLMout_string.replace("'Grammaticality'", '"Grammaticality"')
#     LLMout_string = LLMout_string.replace("'Fluency'", '"Fluency"')
#     # Sometimes justifications are followed by single quotes, sometimes by double quotes
#     LLMout_string = LLMout_string.replace("'Justification': '", '"Justification": "').replace("'Justification'", '"Justification"')
#     LLMout_string = LLMout_string.replace("', 'Score'", '", "Score"').replace("'Score'", '"Score"')
#     if LLMout_string == 'None':
#       eval_missing = dp['id']
#     else:
#       # print(LLMout_string)
#       # LLMout_string = re.sub('("No-Omissions":[^\{]+\{"Justification":)([^\}]+)("Score":[^\}]+\})', '\g<1> "", \g<3>',  LLMout_string)
#       LLMout_string, justifNoOm = separateJustification(LLMout_string, 'No-Omissions')
#       justifications.append(justifNoOm)
#       LLMout_string, justifNoAdd = separateJustification(LLMout_string, 'No-Additions')
#       justifications.append(justifNoAdd)
#       LLMout_string, justifGram = separateJustification(LLMout_string, 'Grammaticality')
#       justifications.append(justifGram)
#       LLMout_string, justifFlu = separateJustification(LLMout_string, 'Fluency')
#       justifications.append(justifFlu)
#       # print(justifications)
#       LLMout_string = LLMout_string.replace("'1'", '"1"')
#       LLMout_string = LLMout_string.replace("'2'", '"2"')
#       LLMout_string = LLMout_string.replace("'3'", '"3"')
#       LLMout_string = LLMout_string.replace("'4'", '"4"')
#       LLMout_string = LLMout_string.replace("'5'", '"5"')
#       LLMout_string = LLMout_string.replace("'6'", '"6"')
#       LLMout_string = LLMout_string.replace("'7'", '"7"')
#       LLMout_string = auto_close_json(LLMout_string)
#       # scores_json = json.loads(LLMout_string)
#       try:
#         scores_json = json.loads(LLMout_string)
#       except Exception:
#           try:
#               scores_json = ast.literal_eval(LLMout_string)
#           except Exception as e:
#               print("\n=== JSON LOAD FAIL ===")
#               print("Offending string:\n", LLMout_string)
#               print("=====================\n")
#               raise e
#       clean_scores_json = None
#       # Gemini adds a node "query" in the json, unlike OpenAI's models
#       if 'query' in scores_json:
#         clean_scores_json = scores_json['query']
#       else:
#         clean_scores_json = scores_json

#       gram_score = int(clean_scores_json['Grammaticality']['Score'])
#       flu_score = int(clean_scores_json['Fluency']['Score'])
#       no_om_score = int(clean_scores_json['No-Omissions']['Score'])
#       no_ad_score = int(clean_scores_json['No-Additions']['Score'])

#       if (gram_score not in model_scores) or (flu_score not in model_scores) or (no_om_score not in model_scores) or (no_ad_score not in model_scores):
#         wrong_score = dp['id']

#       formatted_scores["eid"] = dp['id']
#       formatted_scores["annotator_id"] = str(zip_model)
#       formatted_scores["no-omissions"] = no_om_score
#       formatted_scores["no-additions"] = no_ad_score
#       formatted_scores["grammaticality"] = gram_score
#       formatted_scores["fluency"] = flu_score

#       # print(f"Gram: {gram_score}; Flu: {flu_score}; NoOm: {no_om_score}; NoAd: {no_ad_score}.")
#       # print('')

#   return formatted_scores, eval_missing, wrong_score

# # print(path_dir_unzipped)
# # print(model_prefix)
# eval_files = glob.glob(os.path.join(path_dir_unzipped, '*'))
# evals_missing = []
# wrong_scores = []
# all_scores = []
# for filepath in eval_files:
#   # print(filepath)
#   dbfile_x = open(filepath, 'rb')
#   formatted_scores, eval_missing, wrong_score = loadDataPoint(dbfile_x, model_prefix)
#   if eval_missing != None:
#     evals_missing.append(eval_missing)
#   if wrong_score != None:
#     wrong_scores.append(wrong_score)
#   dbfile_x.close()
#   all_scores.append(formatted_scores)
# # print(f'Missing evaluations: {evals_missing}')
# # print(f'Wrong scores: {wrong_scores}')

# # Save all scores into a json file
# path_json_out = zip_language+'_'+zip_model+'_scores.json'
# with codecs.open(path_json_out, 'w', 'utf-8') as outfile:
#   json.dump(all_scores, outfile)

In [ ]:
# import json
# from collections import defaultdict

# # Load your scores file
# with open('EN_o3_scores.json', 'r', encoding='utf-8') as f:
#     scores = json.load(f)

# # Mapping: model_name -> list of scores (for each dimension)
# models = ['agent', 'e2e', 'struct', 'human']
# dimensions = ['no-omissions', 'no-additions', 'grammaticality', 'fluency']

# # Set up structure
# model_scores = {model: {dim: [] for dim in dimensions} for model in models}

# for entry in scores:
#     eid = entry.get('eid', '').lower()
#     for model in models:
#         if model in eid:  # crude string matching, works with your naming
#             for dim in dimensions:
#                 val = entry.get(dim)
#                 if isinstance(val, int) or (isinstance(val, str) and val.isdigit()):
#                     model_scores[model][dim].append(int(val))
#             break  # Only assign to one model

# # Now compute means
# import numpy as np

# # print(f"{'Model':<8} {'No-Omissions':>14} {'No-Additions':>14} {'Grammaticality':>16} {'Fluency':>10}")
# print(f"{'Model':<8} {'Fluency':>12} {'Grammaticality':>16} {'No-Additions':>14} {'No-Omissions':>12}")
# for model in models:
#     row = [model]
#     for dim in dimensions:
#         vals = model_scores[model][dim]
#         avg = np.mean(vals) if vals else float('nan')
#         row.append(f"{avg:.2f}")
#     print(f"{row[0]:<6} {row[4]:>12} {row[3]:>14} {row[2]:>16} {row[1]:>10}")
